# Lab 2.4 - Function Calling and a Simple Agent (Solution)
**Day 2 - Session 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/lab2_4_agent/solution/lab2_4_agent.ipynb)

**Goal:** Give a model tools it can call, then let a smolagents `CodeAgent`
chain those tools to answer a question. Watch the ReAct loop of thought,
action, and observation. The stretch goal rebuilds the same idea in LangGraph.

This solution has the full working code. It needs an LLM API key (set in Colab
or your local `.env`). Runs on CPU; no GPU needed.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q smolagents langgraph litellm fastapi python-dotenv

## Setup: make the shared client available
This finds the `shared/` package locally, or clones the companion repo when
running on Colab. It also makes sure an API key is set.

In [ ]:
import os
import subprocess
import sys


def _find_repo_root():
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, "shared")):
            return d
        d = os.path.dirname(d)
    return None


root = _find_repo_root()
if root is None:  # On Colab: clone the companion repo for the shared client.
    if not os.path.isdir("gen-ai-companion-intuite"):
        subprocess.run(
            ["git", "clone", "-q",
             "https://github.com/kpassoubady/gen-ai-companion-intuite.git"],
            check=True,
        )
    root = "gen-ai-companion-intuite"
sys.path.insert(0, root)

from shared import get_completion_full, get_model, show_config

if not any(os.environ.get(k) for k in ("OPENAI_API_KEY", "GEMINI_API_KEY",
                                       "ANTHROPIC_API_KEY", "AZURE_API_KEY")):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
    os.environ.setdefault("LLM_PROVIDER", "openai")

show_config()

## 1. Define two tools
A tool is a plain function with a docstring and typed arguments. The `@tool`
decorator exposes it to the agent. The model reads the name and docstring to
decide when to call it.

In [ ]:
from smolagents import tool


@tool
def get_tax_rate(state_code: str) -> float:
    """Return the sales tax rate as a percent for a US state.

    Args:
        state_code: Two-letter state code, for example CA or NY.
    """
    rates = {"CA": 7.25, "NY": 4.0, "TX": 6.25, "WA": 6.5, "FL": 6.0}
    return rates.get(state_code.upper(), 0.0)


@tool
def calculate_total(amount: float, tax_rate_percent: float) -> float:
    """Add sales tax to an amount and return the total.

    Args:
        amount: The pre-tax amount in dollars.
        tax_rate_percent: The tax rate as a percent, for example 7.25.
    """
    return round(amount * (1 + tax_rate_percent / 100.0), 2)


print("Sanity check:", calculate_total(250.0, get_tax_rate("CA")))

## 2. Build a CodeAgent
`LiteLLMModel` routes to whatever provider the shared config selects, so the
agent uses the same model as the other labs. We cap `max_steps` so the loop
cannot run away.

In [ ]:
from smolagents import CodeAgent, LiteLLMModel

model = LiteLLMModel(model_id=get_model("mini"))
agent = CodeAgent(
    tools=[get_tax_rate, calculate_total],
    model=model,
    max_steps=5,
)

## 3. Run the agent
The agent reasons, calls `get_tax_rate`, then `calculate_total`, and returns
the answer. The printed log shows each thought, action, and observation.

In [ ]:
answer = agent.run(
    "What is the total cost of a $250 purchase in California after sales tax?"
)
print("\nFinal answer:", answer)

The total should be about 268.12 (250 plus 7.25% tax). The log above is the
ReAct loop in action: think, act with a tool, observe the result, repeat.

## Stretch goals
1. Add a third tool (for example a currency converter) and ask a question that
   needs all three.
2. Rebuild the agent with LangGraph and compare the two styles.

### Stretch 2: the same idea in LangGraph
smolagents hides the loop. LangGraph makes it explicit as a graph: an `agent`
node calls the model, a `tools` node runs any requested tool, and a condition
decides whether to loop or stop. The model call still goes through the shared
client (`get_completion_full`), passing the tool schemas along.

In [ ]:
import json

from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict

TOOLS = {"get_tax_rate": get_tax_rate, "calculate_total": calculate_total}
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_tax_rate",
            "description": "Return the sales tax rate as a percent for a US state.",
            "parameters": {
                "type": "object",
                "properties": {"state_code": {"type": "string"}},
                "required": ["state_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_total",
            "description": "Add sales tax to an amount and return the total.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number"},
                    "tax_rate_percent": {"type": "number"},
                },
                "required": ["amount", "tax_rate_percent"],
            },
        },
    },
]


class State(TypedDict):
    messages: list


def agent_node(state):
    response = get_completion_full(
        messages=state["messages"],
        tier="mini",
        tools=TOOL_SCHEMAS,
    )
    return {"messages": state["messages"] + [response.choices[0].message.model_dump()]}


def tools_node(state):
    last = state["messages"][-1]
    results = []
    for call in last["tool_calls"]:
        name = call["function"]["name"]
        args = json.loads(call["function"]["arguments"])
        output = TOOLS[name](**args)
        results.append({
            "role": "tool",
            "tool_call_id": call["id"],
            "content": str(output),
        })
    return {"messages": state["messages"] + results}


def should_continue(state):
    return "tools" if state["messages"][-1].get("tool_calls") else END


graph = StateGraph(State)
graph.add_node("agent", agent_node)
graph.add_node("tools", tools_node)
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")
app = graph.compile()

result = app.invoke({"messages": [
    {"role": "user",
     "content": "What is the total for a $250 purchase in California after tax?"}
]})
print("LangGraph final answer:", result["messages"][-1]["content"])

## Expected output
- The sanity check prints about 268.12.
- The CodeAgent log shows it calling `get_tax_rate` then `calculate_total`.
- The final answer is about $268.12.
- The LangGraph version reaches the same total through an explicit graph loop.